<div class='alert alert-warning'>

SciPy's interactive examples with Jupyterlite are experimental and may not always work as expected. Execution of cells containing imports may result in large downloads (up to 60MB of content for the first import from SciPy). Load times when importing from SciPy may take roughly 10-20 seconds. If you notice any problems, feel free to open an [issue](https://github.com/scipy/scipy/issues/new/choose).

</div>

We use data from https://data.giss.nasa.gov/gistemp/ for global temperature
anomalies, i.e., deviations from the corresponding 1951-1980 means (Combined
Land-Surface Air and Sea-Surface Water Temperature Anomalies, Land-Ocean Temperature
Index, L-OTI).
We use weights to indicate where to inter- and extrapolate the missing data.


In [ ]:
from pathlib import Path
import numpy as np
import scipy
from scipy.signal import whittaker_henderson
# For the most recent data, use
# fname="https://data.giss.nasa.gov/gistemp/tabledata_v4/GLB.Ts+dSST.csv"
# Here, we instead use a copy made in 2026.
fname = Path(scipy.signal.__file__).parent / "tests/data/GLB.Ts+dSST.csv"
data = np.genfromtxt(
    fname=fname,
    delimiter=",", skip_header=2, missing_values="***"
)
year = data[:, 0]
temperature = data[:, 1:13].ravel()  # monthly temperature anomalies
w = np.ones_like(temperature)
# We might have some nan values.
np.sum(np.isnan(temperature))

np.int64(10)

In [ ]:
w[np.isnan(temperature)] = 0
res = whittaker_henderson(temperature, weights=w)
temperature[:5]

array([-0.19, -0.25, -0.09, -0.16, -0.1])

In [ ]:
res.x[:5]

array([-0.18244619, -0.17823282, -0.17409373, -0.17080896, -0.16833158])

Let us plot measurements and Whittaker-Henderson smoothing.


In [ ]:
import matplotlib.pyplot as plt
x = year[0] + np.arange(len(temperature)) / 12
plt.plot(x, temperature, label="measurement")
plt.plot(x, res.x, label="WH smooth")
# Above, we set w = 0 for nan values of temperature. WH automatically
# inter- or extrapolates for all data points with w = 0.
plt.plot(x[w==0], res.x[w==0], color="red", label="inter-/extrapolation")
plt.xlabel("year")
plt.ylabel("temperature deviation [°C]")
plt.title("Global Temperature Anomalies (ref. 1951-1980)")
plt.legend()
plt.show()

We can see that extrapolation has occurred at the right end of the signal, meaning
that NaNs existed in the data for the most recent dates, in particular months of
2026 that have not yet happened at the time of the data download in March 2026.
